# 🧪 Taller Práctico: Transformers, BERT y RoBERTa
 Usaremos modelos preentrenados como BERT y RoBERTa para realizar tareas de análisis de sentimientos, clasificación de texto y análisis de representaciones.

## ✅ Parte 1: Uso de `pipeline` para tareas básicas de NLP

In [6]:
from transformers import pipeline

# Clasificación de sentimiento
task = pipeline(model="tabularisai/multilingual-sentiment-analysis")
text = "Me gusta la pelicula"
result = task(text)
print(result)

Device set to use cuda:0


[{'label': 'Positive', 'score': 0.6731139421463013}]


## ✅ Parte 2: Tokenización y extracción de embeddings

In [7]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "Transformers are powerful models for NLP."
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

print("Shape del embedding:", outputs.last_hidden_state.shape)

Shape del embedding: torch.Size([1, 10, 768])


## ✅ Parte 3: Clasificación de texto con RoBERTa y dataset AG News

In [8]:
!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv
!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv

--2025-06-04 02:10:48--  https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 29470338 (28M) [text/plain]
Saving to: ‘train.csv’

train.csv           100%[===================>]  28.10M   148MB/s    in 0.2s    

2025-06-04 02:10:49 (148 MB/s) - ‘train.csv’ saved [29470338/29470338]

--2025-06-04 02:10:49--  https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18

In [9]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score

# 1. Leer CSV
train_df = pd.read_csv("train.csv", header=None)
train_df.columns = ["label", "title", "description"]
train_df["label"] = train_df["label"] - 1  # de 1–4 a 0–3
train_df["text"] = train_df["title"] + " " + train_df["description"]

# 2. Convertir a Dataset de Hugging Face
dataset = Dataset.from_pandas(train_df[["text", "label"]])
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# 3. Tokenizador y modelo
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

# 4. Preprocesamiento
def preprocess(example):
    return tokenizer(example["text"], truncation=True)

tokenized_ds = dataset.map(preprocess, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 5. Métrica de evaluación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/108000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

In [10]:
# 6. Configuración de entrenamiento
args = TrainingArguments(
    output_dir="./results",
    do_eval=True,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    push_to_hub=False,
)

# 7. Entrenador
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"].shuffle(seed=42).select(range(1000)),
    eval_dataset=tokenized_ds["test"].select(range(200)),
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

<ipython-input-10-e982e7840213>:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [4]:
# 8. Entrenar y evaluar
trainer.train()
eval_result = trainer.evaluate()
print(eval_result)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hector-gomez (hector-gomez-pontificia-universidad-cat-lica-del-per-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,1.375400
20,0.986400
30,0.609100
40,0.365500
50,0.265000
60,0.274200


{'eval_loss': 0.3223017752170563, 'eval_accuracy': 0.91, 'eval_runtime': 0.3286, 'eval_samples_per_second': 608.704, 'eval_steps_per_second': 39.566, 'epoch': 1.0}


In [8]:

# Diccionario de clases
label_names = {
    0: "Mundo",
    1: "Deportes",
    2: "Negocios",
    3: "Ciencia/Tecnología"
}

In [6]:
import torch

# Detectar el dispositivo donde está el modelo
device = model.device  # usualmente 'cuda' o 'cpu'

# Texto de prueba en español
texto = "Apple lanza un nuevo iPhone con funciones de inteligencia artificial."

# Tokenizar y mover al mismo dispositivo
inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Inferencia
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
pred_id = logits.argmax().item()

print(f"Predicción: {label_names[pred_id]}")


Predicción: Ciencia/Tecnología


In [10]:
textos = [
    "La inflación afecta gravemente a la economía peruana.",
    "Cristiano Ronaldo marca un gol histórico.",
    "Se lanza un nuevo satélite para estudiar el cambio climático.",
    "Apple presenta un nuevo chip para sus dispositivos móviles."
]

for t in textos:
    inputs = tokenizer(t, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax().item()
    print(f"{t} → {label_names[pred]}")


La inflación afecta gravemente a la economía peruana. → Negocios
Cristiano Ronaldo marca un gol histórico. → Deportes
Se lanza un nuevo satélite para estudiar el cambio climático. → Ciencia/Tecnología
Apple presenta un nuevo chip para sus dispositivos móviles. → Ciencia/Tecnología
